# 06 - Exportacion para Power BI

Notebook para preparar tablas finales limpias para Power BI.

El flujo es sencillo:
- comprobar si cada archivo existe antes de leerlo;
- cargar solo los CSV disponibles;
- limpiar nombres, valores y tipos de forma conservadora;
- exportar los resultados en `powerbi/datasets/`.

No se crea ningun archivo `.pbix`. Solo se preparan los datos.

## 1. Importacion de librerias

In [1]:
from pathlib import Path
import re

import pandas as pd
from IPython.display import display

base_dir = Path('..').resolve()
output_dir = base_dir / 'powerbi' / 'datasets'
output_dir.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
pd.set_option('display.max_colwidth', 80)

## 2. Carga y verificacion de archivos

Primero comprobamos la existencia de cada archivo de entrada. Si falta alguno, el notebook sigue ejecutandose con los demas.

In [2]:
input_files = {
    'movies_clean': base_dir / 'data' / 'processed' / 'movies_clean.csv',
    'genre_distribution': base_dir / 'reports' / 'resultados' / 'genre_distribution.csv',
    'top_rated_movies': base_dir / 'reports' / 'resultados' / 'top_rated_movies.csv',
    'most_rated_movies': base_dir / 'reports' / 'resultados' / 'most_rated_movies.csv',
    'ratings_distribution': base_dir / 'reports' / 'resultados' / 'ratings_distribution.csv',
    'recommendations_examples': base_dir / 'reports' / 'resultados' / 'recommendations_examples.csv',
    'evaluation_summary': base_dir / 'reports' / 'resultados' / 'evaluation_summary.csv',
}

loaded_data = {}
missing_files = []

for dataset_name, file_path in input_files.items():
    if file_path.exists():
        loaded_data[dataset_name] = pd.read_csv(file_path)
        print(f'OK  - {dataset_name}: {file_path.name} ({loaded_data[dataset_name].shape[0]} rows, {loaded_data[dataset_name].shape[1]} cols)')
    else:
        missing_files.append(file_path)
        print(f'MISS - {dataset_name}: {file_path}')

print()
print(f'Loaded datasets: {len(loaded_data)}')
print(f'Missing files: {len(missing_files)}')

OK  - movies_clean: movies_clean.csv (87585 rows, 26 cols)
OK  - genre_distribution: genre_distribution.csv (19 rows, 2 cols)
OK  - top_rated_movies: top_rated_movies.csv (10 rows, 5 cols)
OK  - most_rated_movies: most_rated_movies.csv (10 rows, 5 cols)
OK  - ratings_distribution: ratings_distribution.csv (10 rows, 2 cols)
OK  - recommendations_examples: recommendations_examples.csv (25 rows, 10 cols)
OK  - evaluation_summary: evaluation_summary.csv (5 rows, 7 cols)

Loaded datasets: 7
Missing files: 0


## 3. Limpieza final para Power BI

La limpieza es deliberadamente prudente: se normalizan columnas, se recortan textos, se eliminan duplicados y se ordenan las tablas para que el consumo en Power BI sea mas directo.

In [3]:
def sanitize_column_name(name: str) -> str:
    text = str(name).strip()
    text = re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', text)
    text = text.lower().replace(' ', '_')
    text = re.sub(r'[^a-z0-9_]+', '_', text)
    text = re.sub(r'_+', '_', text).strip('_')
    return text


def clean_text_columns(df: pd.DataFrame) -> pd.DataFrame:
    cleaned = df.copy()
    for column in cleaned.columns:
        if pd.api.types.is_object_dtype(cleaned[column]) or pd.api.types.is_string_dtype(cleaned[column]):
            cleaned[column] = (
                cleaned[column]
                .astype('string')
                .str.replace('\r\n', ' ', regex=False)
                .str.replace('\n', ' ', regex=False)
                .str.replace('\r', ' ', regex=False)
                .str.replace(r'\s+', ' ', regex=True)
                .str.strip()
            )
    return cleaned


def coerce_integer_like_columns(df: pd.DataFrame) -> pd.DataFrame:
    cleaned = df.copy()
    integer_like_names = {'movie_id', 'year', 'shared_genres_count', 'num_recommendations'}

    for column in cleaned.columns:
        if column.endswith('_count') or column in integer_like_names:
            numeric_values = pd.to_numeric(cleaned[column], errors='coerce')
            if numeric_values.dropna().map(lambda value: float(value).is_integer()).all():
                cleaned[column] = numeric_values.astype('Int64')

    return cleaned


def prepare_powerbi_table(df: pd.DataFrame) -> pd.DataFrame:
    cleaned = df.copy()
    cleaned = cleaned.drop_duplicates()
    cleaned.columns = [sanitize_column_name(column) for column in cleaned.columns]
    cleaned = clean_text_columns(cleaned)
    cleaned = cleaned.replace([float('inf'), float('-inf')], pd.NA)
    cleaned = cleaned.dropna(how='all')
    cleaned = coerce_integer_like_columns(cleaned)
    return cleaned


def sort_if_possible(df: pd.DataFrame, columns: list[str], ascending: list[bool]) -> pd.DataFrame:
    existing = [column for column in columns if column in df.columns]
    if not existing:
        return df

    asc = ascending[: len(existing)]
    if len(asc) < len(existing):
        asc = asc + [True] * (len(existing) - len(asc))

    return df.sort_values(by=existing, ascending=asc, kind='mergesort').reset_index(drop=True)


def sort_ratings_distribution(df: pd.DataFrame) -> pd.DataFrame:
    if {'rating_interval', 'movie_count'}.issubset(df.columns):
        ordered = df.copy()
        ordered['_interval_start'] = pd.to_numeric(
            ordered['rating_interval'].astype('string').str.split('-').str[0],
            errors='coerce',
        )
        ordered = ordered.sort_values(by=['_interval_start', 'movie_count'], ascending=[True, False], kind='mergesort')
        ordered = ordered.drop(columns=['_interval_start']).reset_index(drop=True)
        return ordered
    return df


cleaned_data = {}

if 'movies_clean' in loaded_data:
    cleaned_data['movies_clean_powerbi'] = sort_if_possible(
        prepare_powerbi_table(loaded_data['movies_clean']),
        ['rating_count', 'rating_mean', 'title_clean'],
        [False, False, True],
    )

if 'genre_distribution' in loaded_data:
    cleaned_data['genre_distribution_powerbi'] = sort_if_possible(
        prepare_powerbi_table(loaded_data['genre_distribution']),
        ['movie_count', 'genre'],
        [False, True],
    )

if 'top_rated_movies' in loaded_data:
    cleaned_data['top_rated_movies_powerbi'] = sort_if_possible(
        prepare_powerbi_table(loaded_data['top_rated_movies']),
        ['rating_mean', 'rating_count', 'title_clean'],
        [False, False, True],
    )

if 'most_rated_movies' in loaded_data:
    cleaned_data['most_rated_movies_powerbi'] = sort_if_possible(
        prepare_powerbi_table(loaded_data['most_rated_movies']),
        ['rating_count', 'rating_mean', 'title_clean'],
        [False, False, True],
    )

if 'ratings_distribution' in loaded_data:
    cleaned_data['ratings_distribution_powerbi'] = sort_ratings_distribution(
        prepare_powerbi_table(loaded_data['ratings_distribution'])
    )

if 'recommendations_examples' in loaded_data:
    cleaned_data['recommendations_examples_powerbi'] = sort_if_possible(
        prepare_powerbi_table(loaded_data['recommendations_examples']),
        ['input_movie', 'category', 'similarity_score', 'rating_mean', 'title'],
        [True, True, False, False, True],
    )

if 'evaluation_summary' in loaded_data:
    cleaned_data['evaluation_summary_powerbi'] = sort_if_possible(
        prepare_powerbi_table(loaded_data['evaluation_summary']),
        ['category', 'input_movie'],
        [True, True],
    )

print(f'Prepared tables: {len(cleaned_data)}')
for name, df in cleaned_data.items():
    print(f'{name}: {df.shape[0]} rows, {df.shape[1]} cols')

Prepared tables: 7
movies_clean_powerbi: 87585 rows, 26 cols
genre_distribution_powerbi: 19 rows, 2 cols
top_rated_movies_powerbi: 10 rows, 5 cols
most_rated_movies_powerbi: 10 rows, 5 cols
ratings_distribution_powerbi: 10 rows, 2 cols
recommendations_examples_powerbi: 25 rows, 10 cols
evaluation_summary_powerbi: 5 rows, 7 cols


## 4. Exportacion de CSV finales

Los archivos se guardan en `powerbi/datasets/` con codificacion UTF-8-SIG para facilitar su apertura en Power BI y en herramientas de Office.

In [4]:
export_paths = {
    'movies_clean_powerbi': output_dir / 'movies_clean_powerbi.csv',
    'genre_distribution_powerbi': output_dir / 'genre_distribution_powerbi.csv',
    'top_rated_movies_powerbi': output_dir / 'top_rated_movies_powerbi.csv',
    'most_rated_movies_powerbi': output_dir / 'most_rated_movies_powerbi.csv',
    'ratings_distribution_powerbi': output_dir / 'ratings_distribution_powerbi.csv',
    'recommendations_examples_powerbi': output_dir / 'recommendations_examples_powerbi.csv',
    'evaluation_summary_powerbi': output_dir / 'evaluation_summary_powerbi.csv',
}

saved_files = []

for dataset_name, output_path in export_paths.items():
    if dataset_name in cleaned_data:
        cleaned_data[dataset_name].to_csv(output_path, index=False, encoding='utf-8-sig')
        saved_files.append({
            'dataset': dataset_name,
            'file': output_path.name,
            'rows': cleaned_data[dataset_name].shape[0],
            'columns': cleaned_data[dataset_name].shape[1],
        })

saved_summary = pd.DataFrame(saved_files)
display(saved_summary)

if missing_files:
    print('Missing input files:')
    for file_path in missing_files:
        print(f' - {file_path}')
else:
    print('All expected input files were available.')

,dataset,file,rows,columns
0,movies_clean_powerbi,movies_clean_powerbi.csv,87585,26
1,genre_distribution_powerbi,genre_distribution_powerbi.csv,19,2
2,top_rated_movies_powerbi,top_rated_movies_powerbi.csv,10,5
3,most_rated_movies_powerbi,most_rated_movies_powerbi.csv,10,5
4,ratings_distribution_powerbi,ratings_distribution_powerbi.csv,10,2
5,recommendations_examples_powerbi,recommendations_examples_powerbi.csv,25,10
6,evaluation_summary_powerbi,evaluation_summary_powerbi.csv,5,7


All expected input files were available.


## 5. Visualizaciones sugeridas en Power BI

Con estas tablas ya preparadas, en Power BI se pueden crear visualizaciones claras y utiles como:
- distribucion de generos: grafico de barras o treemap para ver que generos concentran mas peliculas;
- top peliculas: tabla o grafico de barras con las peliculas mejor valoradas y su volumen de votos;
- distribucion de valoraciones: histograma o barras por intervalos para entender como se reparten las notas;
- ejemplos de recomendaciones: tabla interactiva con filtros por pelicula de entrada y categoria;
- resumen de evaluacion: tarjetas KPI y tabla resumen con medias de similitud, valoracion y numero de recomendaciones.

El objetivo de este notebook es dejar los datos listos para conectar estas tablas directamente en un modelo de Power BI.